[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/personal-finance-dashboard/personal_finance_dashboard.ipynb)
# 💰 Personal Finance Dashboard

This notebook teaches and demonstrates a beginner-friendly personal finance dashboard using CSV storage, pandas, and Plotly.

## 🎯 Project Overview

The dashboard accepts income and expense transactions, saves them in a CSV file, reads them into a pandas DataFrame, prints a summary, and creates three interactive charts. Amounts are positive US dollar values; the transaction type identifies income or expense.

## 🐍 Python Concepts

This project practices:

- CSV storage with `open()`, `with open(...)`, append mode, and `file.write()`
- `os.path.exists()` and `os.stat()`
- pandas, `pd.read_csv()`, and DataFrames
- DataFrame column selection, `groupby()`, `sum()`, and `reset_index()`
- `datetime.strptime()` and `pd.to_datetime()`
- Plotly Express with `px.bar()`, `px.pie()`, and `px.line()`
- Functions, loops, input validation, and the `if __name__ == "__main__":` guard

## 💾 CSV File Storage

A CSV file stores rows of comma-separated values. Opening a file with mode `"a"` means append: new rows are added after existing rows instead of replacing them. The header is written only when the file is new or empty.

In [ ]:
import os
from datetime import datetime

import pandas as pd
import plotly.express as px

CSV_FILENAME = "finance_data.csv"

def write_transaction_to_file(date, amount, category, transaction_type):
    file_is_new = not os.path.exists(CSV_FILENAME)
    file_is_empty = not file_is_new and os.stat(CSV_FILENAME).st_size == 0

    with open(CSV_FILENAME, "a", encoding="utf-8") as file:
        if file_is_new or file_is_empty:
            file.write("Date,Amount,Category,Type\n")
        file.write(f"{date},{amount},{category},{transaction_type}\n")

def read_transactions_from_file():
    columns = ["Date", "Amount", "Category", "Type"]
    if not os.path.exists(CSV_FILENAME):
        return pd.DataFrame(columns=columns)
    return pd.read_csv(CSV_FILENAME)

## 🐼 Working with pandas

`pd.read_csv()` loads the stored rows into a DataFrame. A DataFrame is a table with named columns. Selecting `df["Amount"]` gives one column, while a condition such as `df["Type"] == "expense"` lets us work with only matching rows.

In [ ]:
transactions = pd.DataFrame({
    "Date": ["2026-09-01", "2026-09-01", "2026-09-02", "2026-09-02", "2026-09-03", "2026-09-03", "2026-09-04", "2026-09-04", "2026-09-05", "2026-09-05", "2026-09-06", "2026-09-07"],
    "Amount": [3000.0, 180.0, 120.0, 90.0, 450.0, 160.0, 220.0, 140.0, 300.0, 100.0, 190.0, 130.0],
    "Category": ["Salary", "Groceries", "Transport", "Dining", "Freelance", "Utilities", "Groceries", "Entertainment", "Side Gig", "Transport", "Groceries", "Dining"],
    "Type": ["income", "expense", "expense", "expense", "income", "expense", "expense", "expense", "income", "expense", "expense", "expense"],
})
transactions

## 📅 Working with Dates

Dates are initially text in a CSV file. `pd.to_datetime()` converts the `Date` column to datetime values so pandas can sort and plot dates in chronological order. The explicit `%Y-%m-%d` format matches the stored date format.

In [ ]:
transactions["Date"] = pd.to_datetime(
    transactions["Date"], format="%Y-%m-%d", errors="coerce"
)
transactions

## ➕ Add Transactions

The local script asks for transactions repeatedly. A date is checked with `datetime.strptime()`, an amount is converted with `float()`, and the type is normalized with `.lower()`. Type `done` at the date prompt to finish. The notebook uses a controlled dataset below so it can run without waiting for interactive input.

## 📊 Financial Summary

The totals use DataFrame column selection and `sum()`. Net balance is total income minus total expenses.

In [ ]:
total_income = transactions.loc[transactions["Type"] == "income", "Amount"].sum()
total_expenses = transactions.loc[transactions["Type"] == "expense", "Amount"].sum()
net_balance = total_income - total_expenses

print(f"Total Income:   ${total_income:,.2f}")
print(f"Total Expenses: ${total_expenses:,.2f}")
print(f"Net Balance:    ${net_balance:,.2f}")

## 📈 Interactive Dashboard

Plotly Express creates Figure objects. Calling `.show()` displays an interactive chart.

### Income vs. Expenses

Group by `Type` and sum `Amount` to compare the two totals.

In [ ]:
totals = transactions.groupby("Type")["Amount"].sum().reset_index()
income_expense_chart = px.bar(
    totals, x="Type", y="Amount", title="Income vs. Expenses"
)
income_expense_chart.show()

### Expenses by Category

Filter to `expense` rows before grouping by `Category`. This prevents income categories from appearing in the pie chart.

In [ ]:
expenses = transactions[transactions["Type"] == "expense"]
category_totals = expenses.groupby("Category")["Amount"].sum().reset_index()
expenses_by_category_chart = px.pie(
    category_totals,
    names="Category",
    values="Amount",
    title="Expenses by Category",
)
expenses_by_category_chart.show()

### Daily Income and Expense Trend

Group by both `Date` and `Type` so multiple transactions on the same day and of the same type become one daily total.

In [ ]:
daily_totals = (
    transactions.groupby(["Date", "Type"])["Amount"].sum().reset_index()
)
daily_totals = daily_totals.sort_values("Date")
daily_trend_chart = px.line(
    daily_totals,
    x="Date",
    y="Amount",
    color="Type",
    markers=True,
    title="Daily Income and Expense Trend",
)
daily_trend_chart.show()

## 🧪 Try It Yourself

Use this controlled in-memory dataset to test the complete dashboard without changing `finance_data.csv` or using any existing personal transaction data.

| Date | Amount | Category | Type |
| --- | ---: | --- | --- |
| 2026-09-01 | 3000 | Salary | income |
| 2026-09-01 | 120 | Groceries | expense |
| 2026-09-02 | 60 | Transport | expense |
| 2026-09-02 | 90 | Dining | expense |
| 2026-09-03 | 500 | Freelance | income |
| 2026-09-03 | 140 | Utilities | expense |
| 2026-09-04 | 180 | Groceries | expense |
| 2026-09-04 | 120 | Entertainment | expense |
| 2026-09-05 | 350 | Side Gig | income |
| 2026-09-05 | 75 | Transport | expense |
| 2026-09-06 | 160 | Groceries | expense |
| 2026-09-06 | 110 | Dining | expense |
| 2026-09-07 | 130 | Utilities | expense |
| 2026-09-08 | 600 | Freelance | income |
| 2026-09-08 | 190 | Groceries | expense |
| 2026-09-09 | 80 | Transport | expense |
| 2026-09-09 | 150 | Entertainment | expense |
| 2026-09-10 | 400 | Side Gig | income |
| 2026-09-10 | 95 | Dining | expense |
| 2026-09-10 | 145 | Utilities | expense |

The next cells calculate the summary, prepare grouped data, and display all three dashboard charts.

Expected Results / What You Should See:

```text
Total Income:   $4,850.00
Total Expenses: $1,845.00
Net Balance:    $3,005.00
```

Expected Expenses by Category:

- Groceries: \$650.00
- Utilities: \$415.00
- Dining: \$295.00
- Entertainment: \$270.00
- Transport: \$215.00

Expected Daily Trend data:

- 2026-09-01 income \$3,000.00; expense \$120.00
- 2026-09-02 expense \$150.00
- 2026-09-03 income \$500.00; expense \$140.00
- 2026-09-04 expense \$300.00
- 2026-09-05 income \$350.00; expense \$75.00
- 2026-09-06 expense \$270.00
- 2026-09-07 expense \$130.00
- 2026-09-08 income \$600.00; expense \$190.00
- 2026-09-09 expense \$230.00
- 2026-09-10 income \$400.00; expense \$240.00

The bar chart should compare income \$4,850 with expenses \$1,845. The pie chart should show Groceries as the largest expense category at \$650. The line chart should show expense activity across all 10 dates, income on 5 dates, income peaking at \$3,000 on 2026-09-01, the largest daily expense total of \$300 on 2026-09-04, and \$240 of expenses on 2026-09-10.

In [ ]:
# Create the controlled sample in memory; this does not write to finance_data.csv
sample_transactions = pd.DataFrame([
    ["2026-09-01", 3000.0, "Salary", "income"],
    ["2026-09-01", 120.0, "Groceries", "expense"],
    ["2026-09-02", 60.0, "Transport", "expense"],
    ["2026-09-02", 90.0, "Dining", "expense"],
    ["2026-09-03", 500.0, "Freelance", "income"],
    ["2026-09-03", 140.0, "Utilities", "expense"],
    ["2026-09-04", 180.0, "Groceries", "expense"],
    ["2026-09-04", 120.0, "Entertainment", "expense"],
    ["2026-09-05", 350.0, "Side Gig", "income"],
    ["2026-09-05", 75.0, "Transport", "expense"],
    ["2026-09-06", 160.0, "Groceries", "expense"],
    ["2026-09-06", 110.0, "Dining", "expense"],
    ["2026-09-07", 130.0, "Utilities", "expense"],
    ["2026-09-08", 600.0, "Freelance", "income"],
    ["2026-09-08", 190.0, "Groceries", "expense"],
    ["2026-09-09", 80.0, "Transport", "expense"],
    ["2026-09-09", 150.0, "Entertainment", "expense"],
    ["2026-09-10", 400.0, "Side Gig", "income"],
    ["2026-09-10", 95.0, "Dining", "expense"],
    ["2026-09-10", 145.0, "Utilities", "expense"],
], columns=["Date", "Amount", "Category", "Type"])
sample_transactions["Date"] = pd.to_datetime(
    sample_transactions["Date"], format="%Y-%m-%d", errors="coerce"
)
sample_transactions

### Sample Dashboard Summary

Calculate total income, total expenses, and net balance from the controlled sample.

In [ ]:
sample_income = sample_transactions.loc[sample_transactions["Type"] == "income", "Amount"].sum()
sample_expenses = sample_transactions.loc[sample_transactions["Type"] == "expense", "Amount"].sum()
sample_net_balance = sample_income - sample_expenses

print(f"Total Income:   ${sample_income:,.2f}")
print(f"Total Expenses: ${sample_expenses:,.2f}")
print(f"Net Balance:    ${sample_net_balance:,.2f}")

### Sample Income vs. Expenses

Group by transaction type and sum the amounts before creating the bar chart.

In [ ]:
sample_totals = sample_transactions.groupby("Type")["Amount"].sum().reset_index()
sample_income_expense_chart = px.bar(
    sample_totals,
    x="Type",
    y="Amount",
    title="Income vs. Expenses",
)
sample_income_expense_chart.show()

### Sample Expenses by Category

Filter to expenses, group by category, and sum the amounts before creating the pie chart.

In [ ]:
sample_expenses_by_category = sample_transactions[sample_transactions["Type"] == "expense"]
sample_category_totals = sample_expenses_by_category.groupby("Category")["Amount"].sum().reset_index()
sample_expenses_by_category_chart = px.pie(
    sample_category_totals,
    names="Category",
    values="Amount",
    title="Expenses by Category",
)
sample_expenses_by_category_chart.show()

### Sample Daily Income and Expense Trend

Group by both `Date` and `Type` so the line chart shows one total for each transaction type on each day.

In [ ]:
sample_daily_totals = (
    sample_transactions.groupby(["Date", "Type"])["Amount"].sum().reset_index()
)
sample_daily_totals = sample_daily_totals.sort_values("Date")
sample_daily_trend_chart = px.line(
    sample_daily_totals,
    x="Date",
    y="Amount",
    color="Type",
    markers=True,
    title="Daily Income and Expense Trend",
)
sample_daily_trend_chart.show()

## 📚 What I Learned

This project connects plain CSV file writing with pandas analysis and Plotly visualization. I practiced validating user input, keeping income and expenses positive, grouping rows, converting dates, and turning summarized data into interactive charts.

## 📝 Notes

- `finance_data.csv` is generated at runtime and is not part of the committed project source.
- Existing rows remain in the CSV between runs. Remove the file when a clean dataset is needed.
- The notebook uses a small in-memory dataset for repeatable chart experiments.